In [5]:
# CELL 1: Install statsmodels
!pip install statsmodels


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# CELL 2: Load Data & Calculate Log Returns
import pandas as pd
import numpy as np

print("Loading Master Dataset...")
df = pd.read_parquet('MTech_Master_Data.parquet')

# ── Validate the parquet is not empty ────────────────────────────────────────
print(f"Parquet loaded: {len(df):,} rows, {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")
print(f"NaN count: {df.isna().sum().sum()}")

if len(df) == 0:
    raise ValueError(
        " Parquet file is EMPTY.\n"
        "   Go back to Data_Ingestion.ipynb and complete Cells 3 & 4 first.\n"
        "   Cell 3 fetches BTC from Binance. Cell 4 saves the merged parquet."
    )

# ── Compute log returns: log(price_t / price_{t-1}) ──────────────────────────
returns_df = pd.DataFrame(index=df.index)

for col in df.columns:
    returns_df[f'{col}_Return'] = np.log(df[col] / df[col].shift(1))

returns_df.dropna(inplace=True)

# ── Sanity check: remove inf values that can appear in log returns ────────────
returns_df.replace([np.inf, -np.inf], np.nan, inplace=True)
returns_df.dropna(inplace=True)

print(f"\n Returns computed: {len(returns_df):,} rows")
print(f"   Date range: {returns_df.index.min()} → {returns_df.index.max()}")
returns_df.head(3)

Loading Master Dataset...
Parquet loaded: 36,344 rows, 5 columns
Columns: ['Nifty_Close', 'SP500_Close', 'Gold_Close', 'USDINR_Close', 'BTC_Close']
NaN count: 0

✅ Returns computed: 36,343 rows
   Date range: 2026-03-19 13:31:00+00:00 → 2026-04-13 19:13:00+00:00


,Nifty_Close_Return,SP500_Close_Return,Gold_Close_Return,USDINR_Close_Return,BTC_Close_Return
Datetime,,,,,
2026-03-19 13:31:00+00:00,0.0,0.000573,0.002643,0.000150,-0.000018
2026-03-19 13:32:00+00:00,0.0,0.000362,-0.002403,-0.000043,-0.001225
2026-03-19 13:33:00+00:00,0.0,-0.000014,-0.001488,0.000054,0.002263


In [ ]:
# CELL 3: Dynamic Causal Feature Selection via Granger Causality
from statsmodels.tsa.stattools import grangercausalitytests
import warnings
warnings.filterwarnings('ignore')

target   = 'Nifty_Close_Return'
features = [col for col in returns_df.columns if col != target]

causal_features = []
p_value_log = {}   # track best p-value per feature for fallback ranking

print(f"Running Granger Causality for target: {target}")
print(f"Testing {len(features)} features × 18 lags each...\n")

for feature in features:
    best_p   = 1.0
    best_lag = 0
    test_data = returns_df[[target, feature]].dropna()

    # Search lags 1 (1 min) → 18 (18 mins) for 1m bar data
    for lag in range(1, 19):
        try:
            result  = grangercausalitytests(test_data, maxlag=[lag], verbose=False)

            # ── BUG FIX: result[lag][0]['ssr_ftest'] is a tuple (F, p, df1, df2)
            # ── Original code used result[lag]['ssr_ftest'] → returned the tuple
            # ── itself instead of the p-value float. Index [1] gets the p-value.
            p_value = result[lag][0]['ssr_ftest'][1]

            if p_value < best_p:
                best_p   = p_value
                best_lag = lag
        except:
            continue

    p_value_log[feature] = (best_p, best_lag)

    # Threshold: p < 0.10 (90% confidence) — suitable for noisy 1m financial data
    if best_p < 0.10:
        print(f" CAUSAL: {feature}  →  lag {best_lag}  (p={best_p:.4f})")
        causal_features.append(feature)
    else:
        print(f" NOISE : {feature}  →  best p={best_p:.4f} at lag {best_lag}")

# ── Always include the target itself ─────────────────────────────────────────
causal_features.append(target)

# ── Fallback: if nothing passed, take top 2 lowest p-value features ──────────
if len(causal_features) == 1:
    print("\n  No causal features found. Using top-2 by p-value as fallback.")
    ranked = sorted(p_value_log.items(), key=lambda x: x[1][0])
    fallback = [ranked[0][0], ranked[1][0]]
    causal_features = fallback + [target]
    print(f"   Fallback features: {fallback}")

print(f"\n{'='*55}")
print(f"Final causal feature set ({len(causal_features)} features):")
for f in causal_features:
    print(f"   • {f}")
print(f"{'='*55}")

# ── Save filtered returns for next notebook ───────────────────────────────────
causal_df = returns_df[causal_features]
causal_df.to_parquet('MTech_Causal_Features.parquet')
print(f"\n Saved causal features → 'MTech_Causal_Features.parquet'")
print(f"   Shape: {causal_df.shape}")

Running Granger Causality for target: Nifty_Close_Return
Testing 4 features × 18 lags each...

❌ NOISE : SP500_Close_Return  →  best p=0.9980 at lag 1
❌ NOISE : Gold_Close_Return  →  best p=0.1940 at lag 5
✅ CAUSAL: USDINR_Close_Return  →  lag 15  (p=0.0000)
✅ CAUSAL: BTC_Close_Return  →  lag 3  (p=0.0345)

Final causal feature set (3 features):
   • USDINR_Close_Return
   • BTC_Close_Return
   • Nifty_Close_Return

💾 Saved causal features → 'MTech_Causal_Features.parquet'
   Shape: (36343, 3)
